## 295. Find Median from Data Stream

Find median at any time in data stream , median is middle value of the sorted sequence (odd seq length) for middle seq length,it's avg. of middle 2 elements

```code
arr = [3,2,5,8,9,10] 

left_max = [3],right_min = [],median = [3]
------------
left_max = [3,2],right_min = [] # 2 < left_max.top[=3]
left_max = [2],right_min = [3] # len(left_max)-len(right_min)>1[=2] : move left to right
median = [2,3]
-------------
left_max = [2],right_min = [3,5] # 5 > left_max.top[=2]
left_max = [3,2],right_min = [5] #len(right_min)>len(left_max) : move right to left
median = [3] # since median is sorted middle value that's why not 2 [3,2,5]
---------------
left_max = [3,2],right_min = [5,8] # 8 > left_max.top[=3]
median = [3,5]
----------------
left_max = [3,2],right_min = [5,8,9] # 9 > left_max.top[=3]
left_max = [5,3,2],right_min = [8,9] #len(right_min)>len(left_max) : move right to left
median = [5]
--------------------
left_max = [5,3,2],right_min = [8,9,10] # 10 > left_max.top[=5]
median = [5,8]

```

```python

"""
In python max_heap is done by pushing -ve value

rules for pushing in heaps
------------------------------------------------------- 
# odd no of element -> left_heap will have 1 more element

1. push to left_max -> if 1st element or current < heap.top [odd element] else in right_heap
2. len(left_heap)-len(right_heap)>1 -> move from left to right [can't have more than 1]
3. len(right_heap)>len(left_heap) -> move from right to left [left will have 1 more]

rule for finding median
---------------------------------------------------
1. even no of element -> avg. of both heap
2. odd no of element -> return left_max.top

"""

def addNum(self, num: int) -> None:
      if not self._left_max_heap or num < -1*self._left_max_heap[0]:
          heapq.heappush(self._left_max_heap,-1*num)
      else:
          heapq.heappush(self._right_min_heap,num)

      if abs(len(self._left_max_heap)-len(self._right_min_heap))>1:
          heapq.heappush(self._right_min_heap,-1*heapq.heappop(self._left_max_heap))
      elif len(self._right_min_heap) > len(self._left_max_heap):
          heapq.heappush(self._left_max_heap,-1*heapq.heappop(self._right_min_heap))


def findMedian(self) -> float:
      if len(self._right_min_heap)==len(self._left_max_heap): # even no of elements
          left_max_heap_top = -1*self._left_max_heap[0]
          right_min_heap_top = self._right_min_heap[0]
          return float(left_max_heap_top+right_min_heap_top)/2
      else: # odd no of elements
          left_max_heap_top = -1*self._left_max_heap[0]
          return float(left_max_heap_top)
```

## 451. Sort Characters By Frequency

```bash
Input: s = "cccaaa"
Output: "aaaccc"
Explanation: Both 'c' and 'a' appear three times, so both "cccaaa" and "aaaccc" are valid answers.
Note that "cacaca" is incorrect, as the same characters must be together.
```

```python

# Approach 1 : Using sorting

1. store counts in a dictionary - items
2. sorted_items = sorted(items, key=lambda x: x[1], reverse=True)
3. put into a list - result.append(char * freq) , use "".join()


# Approach 2 : Using Priority Queue

def frequencySort(self, s: str) -> str:
    counts = Counter(s)
    
    # Build a max-heap (default heapq is a min-heap)
    max_heap = []
    for char, freq in counts.items():
        heapq.heappush(max_heap, (-freq, char))
        
    result = []
    
    # Build the result string
    while max_heap:
        neg_freq, char = heapq.heappop(max_heap)
        # neg_freq is negative, so -neg_freq gives the actual count
        result.append(char * (-neg_freq))
        
    return "".join(result)
```

## 1962. Remove Stones to Minimize the Total

You should apply the following operation exactly k times:
Choose any piles[i] and remove ceil(piles[i] / 2) stones from it.
Notice that you can apply the operation on the same pile more than once.

Return the minimum possible total number of stones remaining after applying the k operations.

```bash
Input: piles = [4,3,6,7], k = 3
Output: 12
Explanation: Steps of a possible scenario are:
- Apply the operation on pile 2. The resulting piles are [4,3,3,7].
- Apply the operation on pile 3. The resulting piles are [4,3,3,4].
- Apply the operation on pile 0. The resulting piles are [2,3,3,4].
The total number of stones in [2,3,3,4] is 12.
```

```python

we remove the max pile to reduce the sum, since we need to insert the modified pile in sorted order - use max heap

# TC - k*log(n)+O(n) [sort+insert] in heap
# (same as if we use list + sort)

def minStoneSum(self, piles: List[int], k: int) -> int:
    max_heap = []
    sum_val = 0
    for pile in piles:
        heapq.heappush(max_heap,-1*pile)
        sum_val +=pile
    for _ in range(k):
        largest = -1*heapq.heappop(max_heap)
        sum_val -= largest
        largest = math.ceil(largest/2)
        sum_val += largest
        heapq.heappush(max_heap,-1*largest)
    return sum_val
```

## 1834. Single-Threaded CPU

Given tasks[i] = [enqueueTime_i, processingTime_i]
- CPU will choose the one with the shortest processing time.
- Once a task is started, the CPU will process the entire task without stopping.

Return the order in which the CPU will process the tasks.


```python
def getOrder(self, tasks: List[List[int]]) -> List[int]:
        # sort the list by start time
        sorted_tasks = []
        n = len(tasks)
        for i in range(n):
            start = tasks[i][0]
            duration = tasks[i][1]
            idx = i
            sorted_tasks.append([start,duration,idx])
        sorted_tasks = sorted(sorted_tasks,key=lambda x:x[0])
        cur_time = 0
        min_heap = [] 
        i = 0
        result = []
        while i<n :
            # no task in queue
            if len(min_heap)==0 and cur_time < sorted_tasks[i][0]:
                cur_time = sorted_tasks[i][0]
            # add to queue for later
            while i<n and sorted_tasks[i][0]<=cur_time: 
                duration,idx = sorted_tasks[i][1],sorted_tasks[i][2]
                heapq.heappush(min_heap,[duration,idx])
                i+=1
            if len(min_heap)>0:
                duration,idx = heapq.heappop(min_heap)
                cur_time+=duration
                result.append(idx)
        
        while len(min_heap)>0:
            duration,idx = heapq.heappop(min_heap)
            result.append(idx)
        
        return result

```

## 1046. Last Stone Weight

On each turn, we choose the heaviest two stones and smash them together. Suppose the heaviest two stones have weights x and y with x <= y. The result of this smash is:

    If x == y, both stones are destroyed, and
    If x != y, the stone of weight x is destroyed, and the stone of weight y has new weight y - x.
At the end of the game, there is at most one stone left.

Return the weight of the last remaining stone. If there are no stones left, return 0.
```bash
Input: stones = [2,7,4,1,8,1]
Output: 1
Explanation: 
We combine 7 and 8 to get 1 so the array converts to [2,4,1,1,1] then,
we combine 2 and 4 to get 2 so the array converts to [2,1,1,1] then,
we combine 2 and 1 to get 1 so the array converts to [1,1,1] then,
we combine 1 and 1 to get 0 so the array converts to [1] then that's the value of the last stone.
```

```python
# TC - n*(n*log(n))
while len(stones)>1:
    stones = sorted(stones)
    a = stones.pop()
    b = stones.pop()
    stones.append(abs(a-b))
return stones[0]

# Using Max Heap - n*3*log(n) ~  n*log(n)

mx_heap = []
for stone in stones:
    heapq.heappush(mx_heap,-1*stone)
# only 1 element should be left
while len(mx_heap)>1:
    a = -1*heapq.heappop(mx_heap)
    b = -1*heapq.heappop(mx_heap)
    heapq.heappush(mx_heap,-1*abs(a-b))
return -1*mx_heap[0]

```

## 347. Top K Frequent Elements

Given an integer array nums and an integer k, return the k most frequent elements. You may return the answer in any order.

```bash
Input: nums = [1,1,1,2,2,3], k = 2
Output: [1,2]

Input: nums = [1], k = 1
Output: [1]

```
```python
def topKFrequent(self, nums: List[int], k: int) -> List[int]:
    min_heap = []
    mp = {}
    for num in nums:
        mp[num] = 1+mp.get(num,0)
    
    for key,val in mp.items():
        heapq.heappush(min_heap,[val,key])
        if len(min_heap)>k:
            heapq.heappop(min_heap)
    result = []
    for val,key in min_heap:
        result.append(key)
    return result

```

## 2542. Maximum Subsequence Score

For chosen indices i0, i1, ..., ik - 1, your score is defined as:

The sum of the selected elements from nums1 multiplied with the minimum of the selected elements from nums2.
It can defined simply as: (nums1[i0] + nums1[i1] +...+ nums1[ik - 1]) * min(nums2[i0] , nums2[i1], ... ,nums2[ik - 1]).
Return the maximum possible score.

```bash
Input: nums1 = [1,3,3,2], nums2 = [2,1,3,4], k = 3
Output: 12
Explanation: 
The four possible subsequence scores are:
- We choose the indices 0, 1, and 2 with score = (1+3+3) * min(2,1,3) = 7.
- We choose the indices 0, 1, and 3 with score = (1+3+2) * min(2,1,4) = 6. 
- We choose the indices 0, 2, and 3 with score = (1+3+2) * min(2,3,4) = 12. 
- We choose the indices 1, 2, and 3 with score = (3+3+2) * min(1,3,4) = 8.
Therefore, we return the max score, which is 12.

Input: nums1 = [4,2,3,1,1], nums2 = [7,5,10,9,6], k = 1
Output: 30
Explanation: 
Choosing index 2 is optimal: nums1[2] * nums2[2] = 3 * 10 = 30 is the maximum possible score.
```


```python

## Approach-1
"""
solve(i=0,sum_val=0,min_val=∞,count=0)
│
├── TAKE index 0 (1, 2) → solve(1, 1, 2, 1)
│   │
│   ├── TAKE index 1 (3, 1) → solve(2, 4, 1, 2)
│   │   │
│   │   ├── TAKE index 2 (3, 3) → solve(3, 7, 1, 3)
│   │   │   └── ↳ Base case: count=3 → Return 7 * 1 = 7
│   │   │
│   │   └── SKIP index 2 → solve(3, 4, 1, 2)
│   │       │
│   │       ├── TAKE index 3 (2, 4) → solve(4, 6, 1, 3)
│   │       │   └── ↳ Base case: count=3 → Return 6 * 1 = 6
│   │       │
│   │       └── SKIP index 3 → solve(4, 4, 1, 2)
│   │           └── ↳ Base case: i=n → Return 0
│   │
│   └── SKIP index 1 → solve(2, 1, 2, 1)
│       │
│       ├── TAKE index 2 (3, 3) → solve(3, 4, 2, 2)
│       │   │
│       │   ├── TAKE index 3 (2, 4) → solve(4, 6, 2, 3)
│       │   │   └── ↳ Base case: count=3 → Return 6 * 2 = 12 (⭐ MAX)
│       │   │
│       │   └── SKIP index 3 → solve(4, 4, 2, 2)
│       │       └── ↳ Base case: i=n → Return 0
│       │
│       └── SKIP index 2 → solve(3, 1, 2, 1)
│           └── ... (Cannot reach count=3, returns 0)
│
└── SKIP index 0 → solve(1, 0, ∞, 0)
    │
    ├── TAKE index 1 (3, 1) → solve(2, 3, 1, 1)
    │   │
    │   ├── TAKE index 2 (3, 3) → solve(3, 6, 1, 2)
    │   │   │
    │   │   ├── TAKE index 3 (2, 4) → solve(4, 8, 1, 3)
    │   │   │   └── ↳ Base case: count=3 → Return 8 * 1 = 8
    │   │   │
    │   │   └── SKIP index 3 → solve(4, 6, 1, 2)
    │   │       └── ↳ Return 0
    │   └── ...
    └── ...
"""
# TC - 2^n

def maxScore(self, nums1: List[int], nums2: List[int], k: int) -> int:
    min_heap = []
    n = len(nums1)
    def solve(i,sum_val,min_val,count):
        if count==k:
            return sum_val*min_val
        if i>=n:
            return 0
        heapq.heappush(min_heap,nums2[i])
        take_i = solve(i+1,sum_val+nums1[i],min_heap[0],count+1)
        min_heap.remove(nums2[i])
        heapq.heapify(min_heap)
        not_take_i = solve(i+1,sum_val,min_val,count)
        return max(take_i,not_take_i)
    
    return solve(0,0,0,0)


```

```python
## TC - n*log(n)

"""
You are given two arrays, nums1 and nums2, and an integer k. You need to choose exactly k indices - 

Score = Sum(selected elements from nums1) * min(selected elements from  nums2)

1. Sorting by nums2 : As we iterate through the sorted list, the current element's nums2 value is guaranteed to be the minimum for any subset we build using the current and previous elements.

2. Greedy Choice with Min-Heap: Once the minimum (nums2) is fixed for a given step, the only variable left to maximize is the sum of nums1. To maximize the score, we want the largest possible k values from nums1 seen so far. A min-heap of size k allows us to efficiently:

- Keep track of the k largest nums1 values encountered
- ensure we always have the best possible sum for the current nums2 as the minimum.

min_heap : The min-heap stores the nums1 values. The top of the heap (min_heap[0]) is the smallest nums1 value currently in your selection of k.

- add the current nums1[i] to the heap and update the running sum_val
- pop the smallest value from the heap this keeps the sum optimized for 
  the current nums2 constraint.
- Once the heap has exactly k elements, the current nums2[i] is the 
  smallest, so calculate the score (sum_val * nums2[i]) 

"""
def maxScore(self, nums1: List[int], nums2: List[int], k: int) -> int:
    min_heap = []
    n = len(nums1)
    arr = []
    for i in range(n):
        arr.append([nums1[i],nums2[i]])
    # descending order sort
    arr = sorted(arr,key=lambda x:x[1],reverse=True)
    
    sum_val = 0
    for i in range(k):
        sum_val += arr[i][0]
        heapq.heappush(min_heap,arr[i][0])
    max_val = sum_val * arr[k-1][1]

    for i in range(k,n):
        min_val = arr[i][1]
        if min_heap[0]<arr[i][0]:
            remove_val = heapq.heappop(min_heap)
            heapq.heappush(min_heap,arr[i][0])
            sum_val = sum_val - remove_val + arr[i][0]

        max_val = max(max_val,sum_val*min_val)

    return max_val
```

## 2462. Total Cost to Hire K Workers

You are given a 0-indexed integer array costs where costs[i] is the cost of hiring the ith worker.

You are also given two integers k and candidates. We want to hire exactly k workers according to the following rules:

- You will run k sessions and hire exactly one worker in each session.
- In each hiring session, choose the worker with the lowest cost from either the first candidates workers or the last candidates workers. Break the tie by the smallest index.
- For example, if costs = [3,2,7,7,1,2] and candidates = 2, then in the first hiring session, we will choose the 4th worker because they have the lowest cost [3,2,7,7,1,2].
- In the second hiring session, we will choose 1st worker because they have the same lowest cost as 4th worker but they have the smallest index [3,2,7,7,2]. Please note that the indexing may be changed in the process.
- If there are fewer than candidates workers remaining, choose the worker with the lowest cost among them. Break the tie by the smallest index.
- A worker can only be chosen once.
Return the total cost to hire exactly k workers.
```bash

Input: costs = [17,12,10,2,7,2,11,20,8], k = 3, candidates = 4
Output: 11
Explanation: We hire 3 workers in total. The total cost is initially 0.
- In the first hiring round we choose the worker from [17,12,10,2,7,2,11,20,8]. The lowest cost is 2, and we break the tie by the smallest index, which is 3. The total cost = 0 + 2 = 2.
- In the second hiring round we choose the worker from [17,12,10,7,2,11,20,8]. The lowest cost is 2 (index 4). The total cost = 2 + 2 = 4.
- In the third hiring round we choose the worker from [17,12,10,7,11,20,8]. The lowest cost is 7 (index 3). The total cost = 4 + 7 = 11. Notice that the worker with index 3 was common in the first and last four workers.
The total hiring cost is 11.
```

```python
"""
This problem is about making a series of locally optimal decisions to achieve a globally optimal result. You need to hire k workers, and in each step, you must pick the worker with the lowest cost from either the "front" or the "back" of the list - The challenge is that the "candidates" for hiring are always shifting. As you hire someone from the left, a new person moves into the "candidate pool" from the middle

- By using two min-heaps, we can instantly access the cheapest worker currently available on either side of the remaining unhired group.

1. Refill : If pools are not full or the pointers i and j meet
2. Compare from both pq1,pq2 & take the smaller cost one

Imagine the list as a bridge:

[...left_pool...]   [...unhired_middle...]   [...right_pool...]

- As you hire from left_pool, you increment i to bring the next person from the middle into the pool.
- As you hire from right_pool, you decrement j to bring the next person from the middle into the pool.
- The i <= j check ensures you never hire the same person twice.

"""
# TC : O((k+m)⋅logm) 
# SC : O(m)

def totalCost(self, costs: list[int], k: int, candidates: int) -> int:
    n = len(costs)
    pq1 = []  # Min-heap for the prefix candidates
    pq2 = []  # Min-heap for the suffix candidates
    ans = 0
    hired = 0
    i = 0
    j = n - 1
    # We continue until we have hired k workers
    while hired < k:
        # Fill pq1 with up to 'candidates' elements from the left
        while len(pq1) < candidates and i <= j:
            heapq.heappush(pq1, costs[i])
            i += 1
        # Fill pq2 with up to 'candidates' elements from the right
        while len(pq2) < candidates and j >= i:
            heapq.heappush(pq2, costs[j])
            j -= 1
        # Get the smallest available costs from both sides
        a = pq1[0] if pq1 else float('inf')
        b = pq2[0] if pq2 else float('inf')
        # Pick the smaller cost. If equal, pick from pq1 (left side)
        if a <= b:
            ans += heapq.heappop(pq1)
        else:
            ans += heapq.heappop(pq2)
        hired += 1
        
    return ans
```